# RLHF Part 1: Reward Modeling from Human Preferences

Reinforcement Learning from Human Feedback (RLHF) has two major stages: training a reward model from preference data, then using that reward model to fine-tune the policy via RL. This notebook covers the reward modeling stage.

## Why Reward Modeling

Instruction tuning teaches format but not values. A model trained only on (instruction, response) pairs learns to produce responses that look like good responses — but it cannot distinguish between a helpful and harmless response and a confident-sounding harmful one if both appear in the training data.

RLHF adds a signal about *which* responses are preferred: human annotators compare pairs of model outputs and indicate which they prefer. A reward model trained on these comparisons learns a scalar quality function. The language model is then trained to maximize this reward.

## The Bradley-Terry Model

Reward model training uses the Bradley-Terry model for pairwise preferences. Given two responses y_w (winner) and y_l (loser) to the same prompt x, the probability of the winner being preferred is:

```
P(y_w > y_l | x) = sigmoid(r(x, y_w) - r(x, y_l))
```

The loss function is:
```
L = -log(sigmoid(r(x, y_w) - r(x, y_l)))
```

This is equivalent to a binary cross-entropy loss on the score difference. The model learns to assign higher reward to preferred responses without needing absolute reward values.

In [ ]:
import math
import random
from dataclasses import dataclass

@dataclass
class PreferencePair:
    prompt: str
    chosen: str
    rejected: str
    source: str = 'human'

def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

def bradley_terry_loss(reward_chosen: float, reward_rejected: float) -> float:
    return -math.log(sigmoid(reward_chosen - reward_rejected) + 1e-8)

class RewardModel:
    def __init__(self, vocab_size: int = 1000, seed: int = 42):
        rng = random.Random(seed)
        # Simplified: bag-of-words linear model
        self.weights = {i: rng.gauss(0, 0.01) for i in range(vocab_size)}
        self.lr = 0.01

    def _tokenize(self, text: str) -> list:
        # Simple hash-based tokenization
        return [hash(w) % 1000 for w in text.lower().split()]

    def score(self, prompt: str, response: str) -> float:
        tokens = self._tokenize(prompt + ' ' + response)
        return sum(self.weights.get(t, 0.0) for t in tokens) / max(1, len(tokens))

    def train_step(self, pair: PreferencePair) -> float:
        r_w = self.score(pair.prompt, pair.chosen)
        r_l = self.score(pair.prompt, pair.rejected)
        loss = bradley_terry_loss(r_w, r_l)
        # Gradient update (simplified)
        grad = sigmoid(r_w - r_l) - 1.0
        for t in self._tokenize(pair.prompt + ' ' + pair.chosen):
            self.weights[t] = self.weights.get(t, 0) - self.lr * grad
        for t in self._tokenize(pair.prompt + ' ' + pair.rejected):
            self.weights[t] = self.weights.get(t, 0) + self.lr * grad
        return loss

    def accuracy(self, pairs: list) -> float:
        correct = sum(1 for p in pairs
                      if self.score(p.prompt, p.chosen) > self.score(p.prompt, p.rejected))
        return correct / len(pairs) if pairs else 0.0

# Training data
train_pairs = [
    PreferencePair('What is 2+2?', 'The answer is 4.', 'I think it might be around 3 or so.'),
    PreferencePair('Explain photosynthesis', 'Plants convert sunlight, CO2, and water into glucose and oxygen.', 'Photosynthesis is a thing plants do.'),
    PreferencePair('How do I sort a list in Python?', 'Use list.sort() or sorted(list) — both sort in ascending order by default.', 'You can use various methods.'),
    PreferencePair('What causes rain?', 'Rain forms when water vapor condenses into droplets in clouds and falls due to gravity.', 'Rain is water falling from the sky.'),
    PreferencePair('Summarize quantum mechanics', 'Quantum mechanics describes particle behavior at atomic scales using wave functions and probabilistic measurements.', 'It is a complicated physics thing.'),
]

rm = RewardModel()
print(f'Initial accuracy: {rm.accuracy(train_pairs):.0%}')
for epoch in range(5):
    total_loss = sum(rm.train_step(p) for p in train_pairs)
    acc = rm.accuracy(train_pairs)
    print(f'Epoch {epoch+1}: loss={total_loss:.4f}, accuracy={acc:.0%}')

## Preference Data Quality

The reward model is only as good as the preference data it trains on. Key quality issues:

**Annotation inconsistency**: annotators disagree on subtle cases. High annotator disagreement on a pair signals ambiguity and those pairs often hurt more than they help.

**Position bias in annotation**: annotators often prefer the first response shown to them, similar to LLM-as-judge positional bias.

**Length bias**: annotators prefer longer, more detailed responses regardless of accuracy — the same verbosity bias observed in LLM judges.

**Annotator expertise**: safety judgments require domain expertise. A non-expert annotator may prefer a confident-sounding wrong answer to a hedged correct one.

The Anthropic HH (Helpful and Harmless) dataset and OpenAI's preference data both acknowledge these issues and use filtering, calibration, and consensus-based labeling to mitigate them.

## Reward Model Evaluation

A reward model's quality is measured by:
- **Held-out preference accuracy**: what fraction of held-out pairs does the RM rank correctly?
- **Reward distribution**: does the RM produce meaningful score spread, or does it score everything similarly?
- **Correlation with human judgments on new tasks**: does the RM generalize beyond training distribution?

A reward model that achieves 70-75% accuracy on held-out preference pairs is considered good. Accuracy above 80% often indicates overfitting to specific annotation artifacts rather than genuine quality modeling.

Reward model collapse — where the RL policy finds reward-maximizing outputs that are not actually good — is a major practical problem, addressed in the next notebook on PPO fine-tuning.